### Load all packages

In [42]:
#import sys
#!"{sys.executable}" -m pip install nltk

In [21]:
import pandas as pd
import ast
import numpy as np
import json
import nltk
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download("punkt_tab")
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("stopwords")



[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\linod\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\linod\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\linod\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\linod\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\linod\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

### Create Corpus of Old ChatGPT and human responses from the .jsonl file

In [2]:
rows = []

with open("all.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        
        rows.append({
            "Question": obj.get("question"),
            "Human": obj.get("human_answers"),
            "ChatGPT_Old": obj.get("chatgpt_answers"),
            "Source": obj.get("source")
        })

chatGPT_old = pd.DataFrame(rows)

### Create question index for taking random sample for ChatGPT prompts


In [3]:
chatGPT_old = chatGPT_old.reset_index(drop=True)
chatGPT_old["Question_ID"] = chatGPT_old.index

chatGPT_old.head()

,Question,Human,ChatGPT_Old,Source,Question_ID
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5,0
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5,1
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5,2
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5,3
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5,4


### Create dataframe of new ChatGPT responses

In [4]:
chatGPT_new = pd.read_csv("chatgptNew_filled.csv")
chatGPT_new = chatGPT_new.drop(columns=["Question"])
chatGPT_new.head()

,Question_ID,ChatGPT_New
0,11458,Sloths aren’t “lazy” so much as perfectly buil...
1,1092,A simple way to think about it is that big com...
2,5193,The short answer is that these percentages are...
3,7288,The brain doesn’t really prefer even numbers o...
4,3198,"The American Dream is the idea that anyone, no..."


### Merge human, old ChatGPT, and new ChatGPT responses together

In [5]:
corpus = chatGPT_old.merge(chatGPT_new, on="Question_ID", how="left")
corpus.head()

,Question,Human,ChatGPT_Old,Source,Question_ID,ChatGPT_New
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5,0,NaN
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5,1,NaN
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5,2,NaN
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5,3,NaN
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5,4,NaN


### Parse corpus and restrict to 2000 documents per group

In [6]:
# keep the 2,000 Question_IDs that have ChatGPT_New
keep_qids = set(chatGPT_new["Question_ID"])

# filter to those questions
df = corpus[corpus["Question_ID"].isin(keep_qids)].copy()

# turn Human lists into real lists, then keep only the first human answer
df["Human"] = df["Human"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["Human"] = df["Human"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

# build matched long corpus
human_df = df[["Question_ID", "Question", "Source", "Human"]].rename(columns={"Human": "Response"})
human_df["Group"] = "Human"

old_ai_df = df[["Question_ID", "Question", "Source", "ChatGPT_Old"]].rename(columns={"ChatGPT_Old": "Response"})
old_ai_df["Group"] = "Old_AI"

new_ai_df = df[["Question_ID", "Question", "Source", "ChatGPT_New"]].rename(columns={"ChatGPT_New": "Response"})
new_ai_df["Group"] = "New_AI"

long_df = pd.concat([human_df, old_ai_df, new_ai_df], ignore_index=True)
long_df = long_df.dropna(subset=["Response"]).reset_index(drop=True)
long_df["doc_id"] = long_df.index
long_df = long_df[["doc_id", "Question_ID", "Question", "Source", "Group", "Response"]]

# Flatten responses that are lists or stringified lists, and remove empty responses
def flatten_response(x):
    if isinstance(x, list):
        return x[0] if len(x) > 0 else None
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return None
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed[0] if len(parsed) > 0 else None
        except:
            pass
        return x
    return str(x) if x is not None else None

long_df["Response"] = long_df["Response"].apply(flatten_response)
long_df = long_df.dropna(subset=["Response"])
long_df["Response"] = long_df["Response"].astype(str).str.strip()
long_df = long_df[long_df["Response"] != ""]

print(f"Length Long DF: {len(long_df)}, Human DF: {len(human_df)}, Old AI DF: {len(old_ai_df)}, New AI DF: {len(new_ai_df)}")

Length Long DF: 5967, Human DF: 2000, Old AI DF: 2000, New AI DF: 2000


<unknown>:1: SyntaxWarning: invalid decimal literal


In [7]:
long_df.head()

,doc_id,Question_ID,Question,Source,Group,Response
0,0,6,Why I can not fabricate a religion that preven...,reddit_eli5,Human,Because you 're a minor and your parents get t...
1,1,7,What has changed that we frequently now throw ...,reddit_eli5,Human,It 's three fold : * Stuff is cheaper to mass ...
2,2,8,magic the gathering What is it . how popular i...,reddit_eli5,Human,"EDIT , Nov 21 : By popular demand , now includ..."
3,3,13,"What is a "" Tor exit node "" or "" Tor node "" Wh...",reddit_eli5,Human,I 'm going to go off the assumption that you d...
4,4,15,Why are we forced to use banks to receive our ...,reddit_eli5,Human,"You 're not . You can use a credit union , or ..."


## Build LIB.csv

In [9]:
LIB = long_df.copy()

LIB["char_count"] = LIB["Response"].str.len()
LIB["token_count"] = LIB["Response"].str.split().str.len()

LIB = LIB[["doc_id", "Question_ID", "Source", "Group", "char_count", "token_count"]]

LIB.to_csv("LIB.csv", sep="|", index=False)

LIB.head()

,doc_id,Question_ID,Source,Group,char_count,token_count
0,0,6,reddit_eli5,Human,183,39
1,1,7,reddit_eli5,Human,715,141
2,2,8,reddit_eli5,Human,9258,1950
3,3,13,reddit_eli5,Human,1591,309
4,4,15,reddit_eli5,Human,90,22


## Build CORPUS.csv

In [19]:
# 1: Tokenize + POS tag per document
def tokenize_doc(row):
    doc_id = row["doc_id"]
    text = row["Response"]

    # basic tokenization
    tokens = nltk.word_tokenize(text)

    # POS tagging
    pos_tags = nltk.pos_tag(tokens)

    rows = []
    for i, (tok, pos) in enumerate(pos_tags):
        term = re.sub(r"[^a-z]", "", tok.lower())

        rows.append({
            "doc_id": doc_id,
            "token_id": i,
            "Question_ID": row["Question_ID"],
            "Source": row["Source"],
            "Group": row["Group"],
            "token_str": tok,
            "term_str": term,
            "pos": pos
        })

    return rows

# 2. Build a CORPUS table with one row per token
all_rows = []

for _, row in long_df.iterrows():
    all_rows.extend(tokenize_doc(row))

CORPUS = pd.DataFrame(all_rows)

# 3. Add a POS group column for easier analysis
def pos_group(pos):
    if pos.startswith("NN"):
        return "NOUN"
    elif pos.startswith("VB"):
        return "VERB"
    elif pos.startswith("JJ"):
        return "ADJ"
    elif pos.startswith("RB"):
        return "ADV"
    else:
        return "OTHER"

CORPUS["pos_group"] = CORPUS["pos"].apply(pos_group)

# 4. Cleanup
CORPUS = CORPUS[CORPUS["term_str"] != ""]

# 5. Save
CORPUS.to_csv("CORPUS.csv", sep="|", index=False)

CORPUS.head()

,doc_id,token_id,Question_ID,Source,Group,token_str,term_str,pos,pos_group
0,0,0,6,reddit_eli5,Human,Because,because,IN,OTHER
1,0,1,6,reddit_eli5,Human,you,you,PRP,OTHER
2,0,2,6,reddit_eli5,Human,'re,re,VBP,VERB
3,0,3,6,reddit_eli5,Human,a,a,DT,OTHER
4,0,4,6,reddit_eli5,Human,minor,minor,JJ,ADJ


## Build VOCAB.csv

In [22]:
ps = PorterStemmer()
stop_words = set(stopwords.words("english"))

# total token count
N = len(CORPUS)

# document frequency
df_counts = CORPUS.groupby("term_str")["doc_id"].nunique()

# raw count + most common POS tags
VOCAB = CORPUS.groupby("term_str").agg(
    n=("term_str", "count"),
    max_pos=("pos", lambda x: x.value_counts().idxmax()),
    max_pos_group=("pos_group", lambda x: x.value_counts().idxmax())
).copy()

# probability
VOCAB["p"] = VOCAB["n"] / N

# information content
VOCAB["i"] = -np.log2(VOCAB["p"])

# df
VOCAB["df"] = df_counts

# dfidf
VOCAB["dfidf"] = VOCAB["df"] * VOCAB["i"]

# porter stem
VOCAB["porter_stem"] = VOCAB.index.to_series().apply(ps.stem)

# stopword flag
VOCAB["stop"] = VOCAB.index.to_series().isin(stop_words)

# reset index so term_str becomes a column
VOCAB = VOCAB.reset_index()

VOCAB.to_csv("VOCAB.csv", sep="|", index=False)

VOCAB.head()

,term_str,n,max_pos,max_pos_group,p,i,df,dfidf,porter_stem,stop
0,a,25889,DT,OTHER,0.029838,5.066690,5332,27015.593260,a,True
1,aapl,2,NNP,NOUN,0.000002,18.726742,2,37.453484,aapl,False
2,aaron,1,NNP,NOUN,0.000001,19.726742,1,19.726742,aaron,False
3,ab,4,NNP,NOUN,0.000005,17.726742,4,70.906968,ab,False
4,aba,1,NNP,NOUN,0.000001,19.726742,1,19.726742,aba,False


## Analysis

In [23]:
CORPUS = pd.read_csv("CORPUS.csv", sep="|")
LIB = pd.read_csv("LIB.csv", sep="|")
VOCAB = pd.read_csv("VOCAB.csv", sep="|")


In [17]:
# LIB Analysis
print(f"Number of documents in LIB: {len(LIB)}")
print(f"Average character count per document: {LIB['char_count'].mean():.2f}")
print(f"List of Features in LIB: {LIB.columns.tolist()}")

Number of documents in LIB: 5967
Average character count per document: 840.56
List of Features in LIB: ['doc_id', 'Question_ID', 'Source', 'Group', 'char_count', 'token_count']


In [ ]:
# Corpus Analysis
print(f"Number of tokens in corpus: {len(CORPUS)}")
print(f"List of columns in CORPUS: {CORPUS.columns.tolist()}")

Number of tokens in corpus: 867643
Columns in CORPUS: ['doc_id', 'token_id', 'Question_ID', 'Source', 'Group', 'token_str', 'term_str', 'pos', 'pos_group']
List of columns in CORPUS: ['doc_id', 'token_id', 'Question_ID', 'Source', 'Group', 'token_str', 'term_str', 'pos', 'pos_group']


In [26]:
# VOCAB Analysis

top20_dfidf = (
    VOCAB[
        (VOCAB["stop"] == False) &
        (VOCAB["term_str"].str.len() > 1)
    ]
    .sort_values("dfidf", ascending=False)[["term_str", "dfidf"]]
    .head(20)
)

print(f"Number of observations in VOCAB: {len(VOCAB)}")
print(f"List of columns in VOCAB: {VOCAB.columns.tolist()}")
top20_dfidf


Number of observations in VOCAB: 25651
List of columns in VOCAB: ['term_str', 'n', 'max_pos', 'max_pos_group', 'p', 'i', 'df', 'dfidf', 'porter_stem', 'stop']


,term_str,dfidf
671,also,17973.914640
12862,like,17486.999830
16456,people,13177.238521
13412,make,12997.043836
15541,one,12786.566535
13694,may,12578.372093
24943,way,11468.439623
23163,time,11464.925614
24345,usually,11015.228267
13515,many,10799.940135
